# RAG benchmark · DeepEval drives cli-agent

This notebook runs **native DeepEval `EvaluationDataset.evals_iterator()`**:
DeepEval supplies each golden, the SDK's observed `DeepEvalTarget.ask()`
answers it, and DeepEval scores the resulting trace. The target SDK agent
sees **only the question**, not the golden answer or cited source evidence.

The judge is an **independent LLM**. The optional selected Knowledge context
is sent to that judge only for retrieval metrics, so approve its data policy.

## 1. Independent target and judge configuration

Target configuration comes from the normal cli-agent SDK; judge configuration
is explicit and independent. Set `OPENROUTER_API_KEY` or `OPENAI_API_KEY`
in your process environment, not in the notebook.

In [ ]:
from pathlib import Path
import asyncio
import sys

WORKSPACE = Path.cwd().resolve()
GOLDENS_FILE = WORKSPACE / "rag-goldens" / "goldens.json"
OUTPUT_DIR = WORKSPACE / "rag-goldens" / "evaluation"

CONFIG_FILE = None             # cli-agent project config or None for default
ADMIN_CONFIG_FILE = None       # cli-agent admin config or None for default
APPROVE_TOOLS = ()             # exact tool names only
CAPTURE_KNOWLEDGE_CONTENT = True
MAX_CASES = 3                 # None = all goldens
SDK_TIMEOUT_SECONDS = 600.0
FRESH_SDK = False             # force target reexecution despite answer cache

JUDGE_PROVIDER = "openrouter" # openrouter, openai, ollama
JUDGE_MODEL = "openai/gpt-4.1"
JUDGE_BASE_URL = None
THRESHOLD = 0.7
JUDGE_MAX_CONCURRENT = 4
FRESH_EVALUATION = False      # disable DeepEval judge cache reuse

## 2. Build a native DeepEval dataset and SDK target

`AgentBuilder.build_deepeval_target()` is supplied by the optional
cli-agent SDK integration. Each uncached ask instantiates a **fresh** SDK
agent. Reference answers are used only to annotate DeepEval's evaluation
trace; they never enter cli-agent prompts or tools.

In [ ]:
sys.path.insert(0, str(WORKSPACE / "flow/rag/generate_goldens/scripts"))
from benchmark_adapter import (
    load_goldens, deep_eval_dataset, cached_response, save_response,
    target_fingerprint, judge_model, benchmark_metrics,
    retrieval_test_cases, source_recall,
)
from cli_agent.sdk import AgentBuilder
from cli_agent.settings.config import ensure_default_config_file

project_config = Path(CONFIG_FILE) if CONFIG_FILE is not None else ensure_default_config_file()
admin_config = Path(ADMIN_CONFIG_FILE) if ADMIN_CONFIG_FILE is not None else None

builder = AgentBuilder()
if admin_config is None:
    builder.with_admin_config()  # load normal system admin policy
else:
    builder.with_admin_config(path=admin_config)
builder.with_project_config(
    path=project_config, workspace=WORKSPACE,
    workspace_access="none", response_format="text",
)
if APPROVE_TOOLS:
    allowed = frozenset(APPROVE_TOOLS)
    async def approve(name, arguments):
        return name in allowed
    builder.with_approval_callback(approve)
target = builder.build_deepeval_target(
    include_knowledge_content=CAPTURE_KNOWLEDGE_CONTENT,
)

goldens = load_goldens(GOLDENS_FILE, MAX_CASES)
by_id = {row["id"]: row for row in goldens}
dataset = deep_eval_dataset(goldens)
judge = judge_model(JUDGE_PROVIDER, JUDGE_MODEL, JUDGE_BASE_URL)
target_sha = target_fingerprint(
    WORKSPACE, project_config, admin_config,
    tuple(APPROVE_TOOLS), CAPTURE_KNOWLEDGE_CONTENT,
)
print(f"{len(dataset.goldens)} Goldens | judge: {JUDGE_PROVIDER}/{JUDGE_MODEL}")

## 3. DeepEval runs **and scores** the target

**This is the only cell that asks the target agent.**
The outer `evals_iterator()` creates an evaluation trace for every golden.
The SDK's `@observe` adapter attaches actual output, reference answer,
and verified Knowledge selection; DeepEval executes the metrics upon leaving
each iteration. Previously saved successful responses are replayed through
the same observed SDK adapter when `FRESH_SDK=False` (no target LLM call).

Synchronous iterator mode is deliberate: it preserves the active DeepEval
trace context across notebook top-level `await`; DeepEval still runs all
judging and reporting.

In [ ]:
from deepeval.evaluate import AsyncConfig, CacheConfig, DisplayConfig

options = {
    "cache_config": CacheConfig(
        use_cache=not FRESH_EVALUATION, write_cache=True,
    ),
    "display_config": DisplayConfig(
        results_folder=str(OUTPUT_DIR / "deepeval"),
        inspect_after_run=False,
    ),
}
responses = {}
for golden in dataset.evals_iterator(
    metrics=benchmark_metrics(judge, THRESHOLD, retrieval=False),
    identifier="rag-answer-quality",
    async_config=AsyncConfig(run_async=False),
    **options,
):
    row = by_id[golden.additional_metadata["golden_id"]]
    prior = cached_response(
        row, output=OUTPUT_DIR, target_fingerprint=target_sha,
        fresh=FRESH_SDK,
    )
    response = await asyncio.wait_for(
        target.ask(
            golden.input, expected_output=golden.expected_output,
            cached_response=prior,
        ),
        timeout=SDK_TIMEOUT_SECONDS,
    )
    if prior is None:
        save_response(row, response, output=OUTPUT_DIR,
                      target_fingerprint=target_sha)
    responses[row["id"]] = response

print(f"DeepEval evaluated {len(responses)} cases")

## 4. Optional native retrieval metrics

Only cases with actual, complete selected Knowledge text are eligible for
faithfulness and contextual metrics. No reference quote is substituted for
an observed chunk. DeepEval handles its own judge parallelism and caching.

In [ ]:
from deepeval import evaluate

retrieval_cases = retrieval_test_cases(goldens, responses)
retrieval_results = None
if retrieval_cases:
    retrieval_results = evaluate(
        test_cases=retrieval_cases,
        metrics=benchmark_metrics(judge, THRESHOLD, retrieval=True)[2:],
        identifier="rag-retrieval-quality",
        async_config=AsyncConfig(max_concurrent=JUDGE_MAX_CONCURRENT),
        **options,
    )
else:
    print("No complete selected Knowledge context; retrieval metrics skipped.")

## 5. Optional deterministic source recall

cli-agent-specific source diagnostics are separate from DeepEval scores.
They show which expected document paths were initially retrieved, considered
as candidates and selected for the final answer.

In [ ]:
import pandas as pd
from IPython.display import display

display(pd.DataFrame([
    {"golden_id": row["id"], **(source_recall(row, responses[row["id"]]) or {})}
    for row in goldens
]))
print("DeepEval reports:", OUTPUT_DIR / "deepeval")

## Notes

The SDK's optional DeepEval integration requires the companion
`cli-agent` PR #312 (install it before running this notebook).
The judge's model/provider settings do not change or override the target
model. Target answers are cached separately from DeepEval metric results;
use `FRESH_SDK=True` if documents/index/external tools/model weights change.
The first answer-quality pass is deliberately sequential for reliable trace
scoping, while the optional retrieval evaluation can run judge calls in
parallel. Trace outputs and DeepEval reports may contain confidential data.